# Building an ML Dashboard

---

In this notebook, we will build a complete **Iris Prediction Dashboard** using Streamlit. The dashboard loads our saved Pipeline, lets users adjust feature values with sliders, and displays the prediction with confidence bars in real time.

We will cover:

- Structuring a Streamlit ML app
- Loading the model with caching
- Building an interactive input form with sliders
- Displaying predictions and confidence scores
- Adding data visualizations (feature distribution, probability chart)
- Prediction history with session state
- The complete runnable script

> ⚠️ **Note:** The complete app is in `dashboard.py` in this folder. Run it with `streamlit run dashboard.py` from the `05_interactive_dashboards/` directory.

---

## 1. App Structure

A well-organized Streamlit ML app follows this pattern:

1. Imports
2. Page configuration
3. Model loading (cached)
4. Sidebar: input controls
5. Main area: predictions and visualizations

Let's build each piece

---

## 2. Page Configuration

Always set the page config as the **first** Streamlit call in your script:

In [ ]:
import streamlit as st

st.set_page_config(
    page_title="Iris Prediction Dashboard",
    page_icon="🌸",
    layout="wide", 
)

| **Parameter** | **What It Does** |
| :--- | :--- |
| `page_title` | Sets the browser tab title |
| `page_icon` | Sets the favicon (emoji or image path) |
| `layout="wide"` | Uses the full browser width instead of a narrow centered column. Better for dashboards |

---

## 3. Loading the Model (Cached)

We use `@st.cache_resource` so the model is loaded **once** and reused across every rerun.

In [ ]:
from pathlib import Path
import joblib

MODEL_PATH = Path(__file__).resolve().parent.parent / "01_model_persistence" / "models" / "iris_pipeline.joblib"
TARGET_NAMES = ["setosa", "versicolor", "virginica"]

@st.cache_resource
def load_model():
    """Load the Iris Pipeline once and cache it."""
    return joblib.load(MODEL_PATH)

Without `@st.cache_resource`, the model would be deserialized from disk every time the user moves a slider. With it, the model stays in the memory, predictions are instantaneous.

---

## 4. Sidebar: Input Controls

In [ ]:
st.sidebar.header("🌸 Input Features")
st.sidebar.markdown("Adjust the sliders to change the flower measurements.")

sepal_length = st.sidebar.slider("Sepal Length (cm)", 4.0, 8.0, 5.1, 0.1)
sepal_width = st.sidebar.slider("Sepal Width (cm)", 2.0, 4.5, 3.5, 0.1)
petal_length = st.sidebar.slider("Petal Length (cm)", 1.0, 7.0, 1.4, 0.1)
petal_width = st.sidebar.slider("Petal Width (cm)", 0.1, 2.5, 0.2, 0.1)

The slider parameters are `label`, `min_value`, `max_value`, `default_value`, `step`. The ranges come from the Iris dataset's known feature distributions.

Each slider returns its current value. When the user moves any slider, the script reruns with the new values.

---

## 5. Making Predictions

In [ ]:
import numpy as np

pipeline = load_model()

# Build the feature array
features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

# Make prediction
prediction_id = int(pipeline.predict(features)[0])
probabilities = pipeline.predict_proba(features)[0]
prediction_name = TARGET_NAMES[prediction_id]
confidence = probabilities[prediction_id]

This is identical to the logic in our FastAPI endpoint: the model doesn't care whether it's called from an API or dashboard.

---

## 6. Displaying the Prediction


Now we build the main area with the prediction results.

In [ ]:
st.title("🌸 Iris Prediction Dashboard")
st.markdown("Predict the species of an Iris flower based on sepal and petal measurements.")
st.divider()

# --- Prediction Result ---
col1, col2, col3 = st.columns(3)

with col1:
    st.metric(label="Predicted Species", value=prediction_name.capitalize())")
with col2:
    st.metric(label="Confidence", value=f"{confidence:.1%}")
with col3:
    st.metric(label="Class ID", value=str(prediction_id))

This creates three side-by-side metric cards that update in real time as the user adjusts sliders.

---

## 7. Probability Bar Chart

A bar chart showing the probability for each class gives much more insight than just the top prediction:

In [ ]:
import pandas as pd
import plotly.express as px

st.subheader("Prediction Probabilities")

prob_df = pd.DataFrame({
    "Species": [name.capitalize() for name in TARGET_NAMES],
    "Probability": probabilities
})

fig = px.bar(
    prob_df,
    x="Species",
    y="Probability",
    color="Species",
    color_discrete_map={
        "Setosa": "lightblue",
        "Versicolor": "lightgreen",
        "Virginica": "salmon    "
    },
    range_y=[0, 1],
    text_auto=".2%",
)
fig.update_layout(showlegend=False, yaxis_title="Probability", xaxis_title="")
st.plotly_chart(fig, use_container_width=True)

When the user moves a slider past a decision boundary, you can visually see the bars shift - one class's probability drops while another's rises. This is incredibly powerful for explaining model behavior to non-technical stakeholders.

---

## 8. Feature Context: Where Does This Sample Sit?

To help the user understand where their input falls relative to real data, we can show the Itis dataset with the current sample highlighted:

In [ ]:
from sklearn.datasets import load_iris

@st.cache_data
def load_iris_data():
    """Load the Iris dataset as a DataFrame (cached)."""
    data = load_iris()
    df = pd.DataFrame(data=data.data, columns=data.feature_names)
    df["species"] = [data.target_names[t] for t in data.target]
    return df

st.divider()
st.subheader("Feature Context")

iris_df = load_iris_data()

# Scatter: petal length vs petal width, colored by species
fig2 = px.scatter(
    iris_df,
    x="petal length (cm)",
    y="petal width (cm)",
    color="species",
    opacity=0.6,
    title="Petal Length vs Petal Width",
)

# Add the current input as a large star marker
fig2.add_scatter(
    x=[petal_length],
    y=[petal_width],
    mode="markers",
    marker=dict(size=18, symbol="star", color="red", line=dict(width=2, color="black")),
    name="Your Input"
)
st.plotly_chart(fig2, use_container_width=True)

This scatter plot shows where the user's input (red star) sits relative to the training data. It makes the model's decision intuitive: *"My input is in the middle of the virginica cluster, so the model predicts virginica."*

---

## 9. Prediction History

Using `@st.session_state`, we can track all predictions the user has made:

In [ ]:
# Initialize history
if "history" not in st.session_state:
    st.session_state.history = []
    
# Add current prediction
if st.sidebar.button("💾 Save Prediction"):
    st.session_state.history.append({
        "sepal_length": sepal_length,
        "sepal_width": sepal_width,
        "petal_length": petal_length,
        "petal_width": petal_width,
        "predicted_species": prediction_name,
        "confidence": f"{confidence:.2%}"
    })
    
# Display history
if st.session_state.history:
    st.divider()
    st.subheader("Prediction History")
    history_df = pd.DataFrame(st.session_state.history)
    st.dataframe(history_df)

This lets users explore different inputs and compare predictions, useful during client demos.

---

## 10. Deployment

Streamlit apps can be deployed for free on **Streamlit Community Cloud:**

1. Push your code to a GitHub repository (it's already there)
2. Go to [share.streamlit.io](https://share.streamlit.io/)
3. Connect your GitHub account
4. Select the repo, branch, and script path (`04_mlops/05_interactive_dashboards/dashboard.py`)
5. Click **Deploy**

You get a free public URL like `https://your-app.streamlit.app`

> 💡 For **freelance delivery**: Share the Streamlit Cloud URL with your client. They can interact with the model immediately - no installation, no setup, just a browser linl.

---

## 11. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **App structure** | Config → Model loading → Sidebar inputs → Main area outputs. |
| `@st.cache_resource` | Load the model once. Predictions are instant on every rerun. |
| **Sidebar sliders** | Feature inputs in the sidebar keep the main area clean. |
| `st.metric()` | Clean, prominent display for the prediction and confidence. |
| **Plotly bar chart** | Probability distribution across classes - visual, interactive, client-friendly. |
| **Scatter + star marker** | Shows where the input sits relative to real data - explains the model's decision |
| `st.session_state` | Track prediction history across interactions. |
| **Streamlit Cloud** | Free deployment from Github. Share a URL, not a codebase. |

---

**Next section:** [Monitoring and Maintenance](../06_monitoring_and_maintenance/) — Keeping your deployed model healthy over time.